# 04 – Satellite-Aware Autoencoder

Domain-enhanced autoencoder with:
- **Orbital phase encoding** (LEO ~90 min cyclical sin/cos features)
- **Eclipse detection** from SNR degradation
- **Physics-based features** (thermal gradients, power estimates P=V×I, gyro/mag magnitude)
- **Domain-weighted MSE loss** (RF/power 1.5×, voltage/current 1.3×, eclipse 1.2×, gyro/mag 0.8×)

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay
from src.utils.preprocessing import (
    generate_synthetic_dataset, preprocess_pipeline, FEATURE_COLUMNS, build_feature_weights
)
from src.models.autoencoder import SatelliteAutoencoder

df = generate_synthetic_dataset(n_samples=120_000, seed=42)
feature_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
train, val, test, scaler, all_feature_cols = preprocess_pipeline(df, feature_cols)

# Use all features: base sensors + orbital + physics + rolling
X_train = train[all_feature_cols].values; X_val = val[all_feature_cols].values
X_test = test[all_feature_cols].values; y_test = test['label'].values

# Domain-weighted loss: sensor criticality weights
feature_weights = build_feature_weights(all_feature_cols)
print(f'Features: {len(all_feature_cols)}  (base {len(feature_cols)} + domain + rolling)')
print(f'Weight range: {feature_weights.min():.1f} – {feature_weights.max():.1f}')

In [ ]:
model = SatelliteAutoencoder(
    input_dim=len(all_feature_cols),
    epochs=50,
    feature_weights=feature_weights,
)
model.fit(X_train, X_val)
metrics = model.evaluate(X_test, y_test)
for k, v in metrics.items(): print(f'  {k}: {v:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(model.train_losses_, label='Train')
if model.val_losses_: ax.plot(model.val_losses_, label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()
plt.tight_layout(); plt.savefig('../results/ae_training_curve.png', dpi=100); plt.show()

In [ ]:
scores = model.anomaly_scores(X_test)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores[y_test==0], bins=60, alpha=0.6, label='Normal', color='steelblue')
ax.hist(scores[y_test==1], bins=60, alpha=0.6, label='Anomaly', color='red')
ax.axvline(model.threshold_, color='orange', linestyle='--', label=f'Threshold ({model.threshold_:.4f})')
ax.set_xlabel('Reconstruction Error'); ax.legend()
plt.tight_layout(); plt.savefig('../results/reconstruction_error_hist.png', dpi=100); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, scores, ax=ax, name='Autoencoder')
plt.tight_layout(); plt.savefig('../results/roc_curve.png', dpi=100); plt.show()